# AUTokens50 — Global Exact-Hash Deduplication Pipeline

**Purpose**

Create a deterministic, global, provenance-aware exact deduplication pipeline for the 57 Parquet files in:

```text
/data/joeyllm_data/AUTokens50_with_hash_simhash
```

The cleaned dataset is written to:

```text
~/hash_output
```

The source dataset is treated as **read-only / immutable**. This notebook never overwrites, renames, deletes, or writes inside the input directory.

The exact-deduplication key is the existing column:

```text
hash
```

The previous project notebook established the exact column contract:

```text
text
hash
simhash
```

The current server dataset has been empirically inspected and its persisted `hash` values are 64 characters long. This notebook does **not** infer the generating hash algorithm from that representation. It consumes the existing `hash` column as the authoritative exact-deduplication key and does not recompute or replace `hash` or `simhash` in the output dataset.

## Deterministic survivor policy

For every duplicate `hash`, exactly one row is retained. The survivor is the earliest row under this total order:

1. `part_0.parquet` before `part_1.parquet` before ... before `part_56.parquet`;
2. within one file, smaller DuckDB `file_row_number` first.

The notebook empirically calibrates DuckDB's `file_row_number` base before using it, rather than assuming whether numbering starts at 0 or 1.

## Pipeline

1. Configuration and package checks.
2. Preflight file/schema validation.
3. DuckDB virtual-metadata calibration.
4. Strong source snapshot with SHA-256 checksums.
5. Global `hash` integrity scan.
6. Global exact-duplicate discovery.
7. Deterministic survivor/removal index construction.
8. Exact-hash decision-contract validation without full-text regrouping.
9. Per-part removal-index materialization.
10. Parallel rewrite into `~/hash_output`.
11. Structural and global deduplication validation.
12. Post-run SHA-256 verification that the source dataset is unchanged.
13. Final audit summary.

**Reference**

- Project reference: `au_tokens50_add_hash_simhash_realtime.ipynb` (existing `text`, `hash`, `simhash` contract and SHA-1 hash-generation logic).
- Lee et al., 2022, *Deduplicating Training Data Makes Language Models Better*, ACL. https://aclanthology.org/2022.acl-long.577/
- Penedo et al., 2024, *The FineWeb Datasets: Decanting the Web for the Finest Text Data at Scale*. https://arxiv.org/abs/2406.17557
- Chen et al., 2025, *Data-Juicer 2.0: Cloud-Scale Adaptive Data Processing for and with Foundation Models*. https://arxiv.org/abs/2501.14755
- Wang et al., 2026, *From Agent Traces to Trust: A Survey of Evidence Tracing and Execution Provenance in LLM Agents*. https://arxiv.org/abs/2606.04990


## Cell 1 — Package availability

**Purpose**

Ensure the notebook kernel has the packages required for global Parquet scanning, exact-hash analysis, Arrow-preserving rewrites, concurrent file processing, and tabular audit output.

The cell installs only packages that are missing. It does not touch the dataset.

**Reference**

- DuckDB Python API and Parquet support: https://duckdb.org/docs/stable/clients/python/reference/ and https://duckdb.org/docs/current/data/parquet/overview
- Apache Arrow / PyArrow Parquet API: https://arrow.apache.org/docs/python/parquet.html


In [1]:
# Purpose: Install only missing runtime dependencies for this notebook.
# Reference: DuckDB Python API and Apache Arrow/PyArrow Parquet documentation.

import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "duckdb": "duckdb>=1.3.0",
    "pyarrow": "pyarrow>=14.0.0",
    "pandas": "pandas>=2.0.0",
    "numpy": "numpy>=1.24.0",
    "tqdm": "tqdm>=4.66.0",
}

missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *missing])
else:
    print("All required packages are already available.")


All required packages are already available.


## Cell 2 — Imports, exact paths, and bounded execution settings

**Purpose**

Define the exact input/output locations and runtime parameters without performing any data transformation.

The source path and output path are fixed to the task specification. The output directory is separate from the immutable input directory.

The Jupyter Pod has been empirically measured with cgroup limits of **32 GiB RAM and 8 CPU cores**. To keep the Jupyter server responsive during large DuckDB operations, this notebook deliberately reserves headroom:

- DuckDB uses at most 4 threads;
- DuckDB is limited to 20 GB of managed memory;
- DuckDB spill files are directed to `~/hash_output/_tmp/duckdb_spill`;
- checksum and rewrite concurrency are capped at 4 workers;
- PyArrow batch decoding inside each rewrite worker is kept single-threaded to avoid nested oversubscription.

`RESUME_EXISTING_OUTPUT` defaults to `False`. This prevents the notebook from silently accepting stale cleaned files from a different run.

**Reference**

- Project task constraint: source dataset is locked/immutable; derived data must be written to a scratch/output directory.
- DuckDB configuration: https://duckdb.org/docs/stable/configuration/overview
- DuckDB workload tuning: https://duckdb.org/docs/current/guides/performance/how_to_tune_workloads
- Data-Juicer 2.0 motivates parallel/adaptive processing for large foundation-model datasets: https://arxiv.org/abs/2501.14755


In [2]:
# Purpose: Define exact paths, column names, survivor policy, and parallel runtime settings.
# Reference: Project task constraints; Data-Juicer 2.0 for scalable parallel data processing.

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import time

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

# ------------------------------------------------------------------
# Exact project paths supplied for this task
# ------------------------------------------------------------------
NOTEBOOK_DIR = (Path.home() / "Notebook").resolve()
INPUT_DIR = Path("/data/joeyllm_data/AUTokens50_with_hash_simhash").resolve()
OUTPUT_DIR = (Path.home() / "hash_output").resolve()
AUDIT_DIR = OUTPUT_DIR / "_audit"
TMP_DIR = OUTPUT_DIR / "_tmp"
REMOVAL_INDEX_DIR = TMP_DIR / "removal_index"
WORK_DB = TMP_DIR / "dedup_work.duckdb"

# Exact existing columns established by the prior notebook.
TEXT_COL = "text"
HASH_COL = "hash"
SIMHASH_COL = "simhash"

EXPECTED_PART_IDS = list(range(57))
EXPECTED_INPUT_FILES = [INPUT_DIR / f"part_{i}.parquet" for i in EXPECTED_PART_IDS]

# ------------------------------------------------------------------
# Execution settings
# ------------------------------------------------------------------
CPU_COUNT = max(1, os.cpu_count() or 1)

# The current Jupyter Pod is limited to 8 CPU cores and 32 GiB RAM.
# Keep explicit headroom for JupyterHub, the kernel, Arrow buffers, and filesystem I/O.
DUCKDB_THREADS = min(4, CPU_COUNT)
DUCKDB_MEMORY_LIMIT = "20GB"
CHECKSUM_WORKERS = min(4, CPU_COUNT)
REWRITE_WORKERS = min(4, CPU_COUNT)
REWRITE_BATCH_SIZE = 100_000
PARQUET_COMPRESSION = "zstd"

DUCKDB_SPILL_DIR = TMP_DIR / "duckdb_spill"
DUCKDB_SPILL_DIR.mkdir(parents=True, exist_ok=True)

# Strict safety default: do not silently reuse existing cleaned data files.
RESUME_EXISTING_OUTPUT = False

# Strong immutability proof: byte-level SHA-256 before and after processing.
VERIFY_SOURCE_FILE_SHA256 = True
SHA256_CHUNK_BYTES = 16 * 1024 * 1024

# Deterministic retention policy accepted for this task.
SURVIVOR_POLICY = "lowest part_number, then lowest file_row_number"

RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()
RUN_START_WALL = time.time()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)
REMOVAL_INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook directory:", NOTEBOOK_DIR)
print("Input directory   :", INPUT_DIR)
print("Output directory  :", OUTPUT_DIR)
print("DuckDB threads    :", DUCKDB_THREADS)
print("DuckDB memory     :", DUCKDB_MEMORY_LIMIT)
print("DuckDB spill dir  :", DUCKDB_SPILL_DIR)
print("Checksum workers  :", CHECKSUM_WORKERS)
print("Rewrite workers   :", REWRITE_WORKERS)
print("Survivor policy   :", SURVIVOR_POLICY)


Notebook directory: /home/jovyan/Notebook
Input directory   : /data/joeyllm_data/AUTokens50_with_hash_simhash
Output directory  : /home/jovyan/hash_output
DuckDB threads    : 32
Checksum workers  : 8
Rewrite workers   : 8
Survivor policy   : lowest part_number, then lowest file_row_number


In [3]:
# Purpose: Remove stale partial files left by the interrupted previous rewrite.
# Reference: The rewrite stage writes only .part_*.parquet.partial before atomic promotion.

from pathlib import Path

output_dir = (Path.home() / "hash_output").resolve()

partial_files = sorted(output_dir.glob(".part_*.parquet.partial"))

print(f"Found {len(partial_files)} stale partial files:")
for path in partial_files:
    print(path)

for path in partial_files:
    path.unlink()

print(f"Removed {len(partial_files)} stale partial files.")

Found 8 stale partial files:
/home/jovyan/hash_output/.part_0.parquet.partial
/home/jovyan/hash_output/.part_1.parquet.partial
/home/jovyan/hash_output/.part_2.parquet.partial
/home/jovyan/hash_output/.part_3.parquet.partial
/home/jovyan/hash_output/.part_4.parquet.partial
/home/jovyan/hash_output/.part_5.parquet.partial
/home/jovyan/hash_output/.part_6.parquet.partial
/home/jovyan/hash_output/.part_7.parquet.partial
Removed 8 stale partial files.


## Cell 3 — Preflight file and schema validation

**Purpose**

Fail early unless the input dataset matches the task exactly enough to execute without guessing:

- `part_0.parquet` through `part_56.parquet` must all exist;
- there must be no unexpected additional `part_*.parquet` files;
- every file must expose the same Arrow schema;
- `text`, `hash`, and `simhash` must exist;
- `text` and `hash` must be Arrow string types because the prior hash contract is text-based;
- the source schema must not contain DuckDB's virtual metadata names `filename` or `file_row_number`, because those names are reserved internally by this notebook;
- existing cleaned output files are rejected unless explicit resume mode is enabled.

This cell reads metadata only and does not modify source files.

**Reference**

- Project reference: the prior hash/SimHash notebook establishes `text`, `hash`, and `simhash`.
- DuckDB virtual Parquet metadata (`filename`, `file_row_number`): https://duckdb.org/docs/current/data/parquet/overview


In [4]:
# Purpose: Validate exact file coverage, source schemas, required columns, and output safety.
# Reference: Prior project notebook and DuckDB Parquet virtual-column documentation.

assert NOTEBOOK_DIR.exists(), f"Notebook directory does not exist: {NOTEBOOK_DIR}"
assert INPUT_DIR.exists(), f"Input directory does not exist: {INPUT_DIR}"
assert INPUT_DIR.is_dir(), f"Input path is not a directory: {INPUT_DIR}"
assert INPUT_DIR != OUTPUT_DIR, "Input and output directories must be different."
assert OUTPUT_DIR not in INPUT_DIR.parents, "Output directory must not be inside the input directory."
assert INPUT_DIR not in OUTPUT_DIR.parents, "Input directory must not be inside the output directory."

missing_files = [p for p in EXPECTED_INPUT_FILES if not p.is_file()]
actual_part_files = sorted(INPUT_DIR.glob("part_*.parquet"), key=lambda p: p.name)
expected_set = {p.resolve() for p in EXPECTED_INPUT_FILES}
unexpected_files = [p for p in actual_part_files if p.resolve() not in expected_set]

assert not missing_files, f"Missing required parquet files: {missing_files}"
assert not unexpected_files, f"Unexpected part_*.parquet files found: {unexpected_files}"
assert len(actual_part_files) == 57, f"Expected exactly 57 parquet files, found {len(actual_part_files)}"

schema_rows = []
reference_schema = None

for part_id, path in tqdm(list(zip(EXPECTED_PART_IDS, EXPECTED_INPUT_FILES)), desc="Preflight schemas"):
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    if reference_schema is None:
        reference_schema = schema
    else:
        assert schema.equals(reference_schema, check_metadata=True), f"Schema mismatch in {path}"

    names = schema.names
    schema_rows.append({
        "part_number": part_id,
        "file": str(path),
        "rows": int(pf.metadata.num_rows),
        "size_bytes": int(path.stat().st_size),
        "num_columns": len(names),
        "has_text": TEXT_COL in names,
        "has_hash": HASH_COL in names,
        "has_simhash": SIMHASH_COL in names,
    })

schema_df = pd.DataFrame(schema_rows)

for required in (TEXT_COL, HASH_COL, SIMHASH_COL):
    assert required in reference_schema.names, f"Required column missing from source schema: {required}"

reserved_virtual_names = {"filename", "file_row_number"}
reserved_collisions = reserved_virtual_names.intersection(reference_schema.names)
assert not reserved_collisions, (
    "Source schema contains names reserved for DuckDB virtual Parquet metadata: "
    f"{sorted(reserved_collisions)}. Stop rather than guessing how to disambiguate them."
)

text_type = reference_schema.field(TEXT_COL).type
hash_type = reference_schema.field(HASH_COL).type
assert pa.types.is_string(text_type) or pa.types.is_large_string(text_type), f"Unexpected text type: {text_type}"
assert pa.types.is_string(hash_type) or pa.types.is_large_string(hash_type), f"Unexpected hash type: {hash_type}"

existing_output_parts = sorted(OUTPUT_DIR.glob("part_*.parquet"))
existing_partial_files = sorted(OUTPUT_DIR.glob(".*.partial"))

if not RESUME_EXISTING_OUTPUT:
    assert not existing_output_parts, (
        "Cleaned output part files already exist. Refusing to mix runs. "
        "Delete/move them manually or set RESUME_EXISTING_OUTPUT=True only for the same run."
    )
    assert not existing_partial_files, (
        "Partial output files already exist. Remove them manually or use explicit resume mode."
    )

print(schema_df[["part_number", "rows", "size_bytes", "num_columns"]].to_string(index=False))
print("\nTotal input rows:", int(schema_df["rows"].sum()))
print("Total input bytes:", int(schema_df["size_bytes"].sum()))
print("Columns:", reference_schema.names)
print("Preflight validation: PASS")


Preflight schemas:   0%|          | 0/57 [00:00<?, ?it/s]

 part_number   rows  size_bytes  num_columns
           0 278106  1899045009           11
           1 374529  2550107485           11
           2 268518  1728088552           11
           3 342427  2255059773           11
           4 259925  2108802133           11
           5 288089  2673303001           11
           6 288631  2657432853           11
           7 295847  2741096826           11
           8 267184  2435898446           11
           9 277099  2622978626           11
          10 244561  2062618830           11
          11 300907  2815574632           11
          12 339321  3024574248           11
          13 384765  3292801876           11
          14 405338  2803557401           11
          15 251768  2175864110           11
          16 349707  2220717095           11
          17 361001  2380738388           11
          18 368851  2327913612           11
          19 336559  2134993354           11
          20 388825  2570410994           11
          

## Cell 4 — Calibrate DuckDB `file_row_number` and verify source-file identity

**Purpose**

Use a tiny synthetic Parquet probe to determine the exact numeric base of DuckDB's `file_row_number` in the installed DuckDB version. The notebook then verifies that DuckDB reports the exact same 57 absolute source paths that Python discovered.

This removes two otherwise unsafe assumptions:

1. whether `file_row_number` starts at 0 or 1;
2. whether DuckDB's `filename` values match the paths used by the survivor policy.

The probe is written only under `~/hash_output/_tmp`.

**Reference**

- DuckDB `read_parquet(..., file_row_number=true, filename=true)`: https://duckdb.org/docs/current/data/parquet/overview
- DuckDB Python relational API: https://duckdb.org/docs/stable/clients/python/relational_api


In [5]:
# Purpose: Empirically calibrate file-row numbering and verify exact DuckDB source-file identities.
# Reference: DuckDB Parquet and Python relational API documentation.

con = duckdb.connect(str(WORK_DB))

# Bound DuckDB resource usage so heavy analytical queries do not monopolize the Pod.
con.execute(f"SET threads = {int(DUCKDB_THREADS)}")
con.execute(f"SET memory_limit = '{DUCKDB_MEMORY_LIMIT}'")
con.execute("SET preserve_insertion_order = false")
con.execute(f"SET temp_directory = '{str(DUCKDB_SPILL_DIR).replace(chr(39), chr(39) * 2)}'")

duckdb_runtime_settings = con.execute("""
    SELECT
        current_setting('threads') AS threads,
        current_setting('memory_limit') AS memory_limit,
        current_setting('temp_directory') AS temp_directory,
        current_setting('preserve_insertion_order') AS preserve_insertion_order
""").fetchdf()

display(duckdb_runtime_settings)


def sql_literal(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


probe_path = TMP_DIR / "file_row_number_probe.parquet"
pq.write_table(pa.table({"probe_value": [10, 20, 30]}), probe_path)

probe_rows = con.execute(
    f"SELECT probe_value, file_row_number "
    f"FROM read_parquet({sql_literal(str(probe_path))}, file_row_number=true) "
    f"ORDER BY probe_value"
).fetchall()

probe_numbers = [int(row[1]) for row in probe_rows]
assert len(probe_numbers) == 3, f"Unexpected file_row_number probe result: {probe_rows}"
assert probe_numbers[1] - probe_numbers[0] == 1 and probe_numbers[2] - probe_numbers[1] == 1, (
    f"file_row_number is not consecutive in the probe: {probe_numbers}"
)
FILE_ROW_NUMBER_BASE = probe_numbers[0]
print("DuckDB file_row_number base detected as:", FILE_ROW_NUMBER_BASE)

INPUT_GLOB = str(INPUT_DIR / "part_*.parquet")
INPUT_SCAN_SQL = (
    f"read_parquet({sql_literal(INPUT_GLOB)}, "
    "filename=true, file_row_number=true, hive_partitioning=false, union_by_name=false)"
)

reported_filenames = {
    str(Path(row[0]).resolve())
    for row in con.execute(f"SELECT DISTINCT filename FROM {INPUT_SCAN_SQL}").fetchall()
}
expected_filenames = {str(p.resolve()) for p in EXPECTED_INPUT_FILES}

assert reported_filenames == expected_filenames, (
    "DuckDB filename metadata does not exactly match the expected 57 input paths. "
    f"Missing from DuckDB={sorted(expected_filenames - reported_filenames)}; "
    f"Unexpected from DuckDB={sorted(reported_filenames - expected_filenames)}"
)

source_map_df = pd.DataFrame({
    "source_file": [str(p.resolve()) for p in EXPECTED_INPUT_FILES],
    "part_number": EXPECTED_PART_IDS,
})
con.register("source_map", source_map_df)

unmapped_rows = con.execute(f"""
    SELECT COUNT(*)
    FROM {INPUT_SCAN_SQL} AS s
    LEFT JOIN source_map AS m
      ON s.filename = m.source_file
    WHERE m.source_file IS NULL
""").fetchone()[0]
assert int(unmapped_rows) == 0, f"Found {unmapped_rows} rows whose source file could not be mapped exactly."

probe_path.unlink(missing_ok=True)
print("DuckDB virtual metadata calibration: PASS")


DuckDB file_row_number base detected as: 0
DuckDB virtual metadata calibration: PASS


## Cell 5 — Strong pre-run source snapshot

**Purpose**

Create a byte-level provenance snapshot of every immutable source Parquet file before cleaning. For each file, the notebook records:

- exact path;
- part number;
- row count;
- file size;
- modification time in nanoseconds;
- SHA-256 checksum.

SHA-256 is computed in parallel with threads. A saved run-state file binds any resumed output to this exact source snapshot and survivor policy.

**Reference**

- Wang et al., 2026, *From Agent Traces to Trust*, motivates process-level provenance, auditability, and recovery rather than validating only final outputs: https://arxiv.org/abs/2606.04990
- Engineering integrity control: cryptographic file checksums are used here to prove the locked source files are byte-identical before and after the run.


In [6]:
# Purpose: Capture a strong SHA-256 source snapshot and bind resume state to exact source bytes.
# Reference: 2026 execution-provenance literature plus standard cryptographic integrity practice.

SOURCE_SNAPSHOT_BEFORE_JSON = AUDIT_DIR / "source_snapshot_before.json"
SOURCE_SNAPSHOT_BEFORE_PARQUET = AUDIT_DIR / "source_snapshot_before.parquet"
RUN_STATE_PATH = AUDIT_DIR / "run_state.json"


def sha256_file(path: Path, chunk_bytes: int = SHA256_CHUNK_BYTES) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def snapshot_one(part_id: int, path: Path) -> dict:
    stat = path.stat()
    pf = pq.ParquetFile(path)
    return {
        "part_number": int(part_id),
        "file": str(path.resolve()),
        "rows": int(pf.metadata.num_rows),
        "size_bytes": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
        "sha256": sha256_file(path) if VERIFY_SOURCE_FILE_SHA256 else None,
    }

snapshot_before = []
with ThreadPoolExecutor(max_workers=CHECKSUM_WORKERS) as executor:
    futures = {
        executor.submit(snapshot_one, part_id, path): part_id
        for part_id, path in zip(EXPECTED_PART_IDS, EXPECTED_INPUT_FILES)
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="SHA-256 source snapshot"):
        snapshot_before.append(future.result())

snapshot_before = sorted(snapshot_before, key=lambda x: x["part_number"])
snapshot_before_df = pd.DataFrame(snapshot_before)

SOURCE_SNAPSHOT_BEFORE_JSON.write_text(
    json.dumps(snapshot_before, indent=2), encoding="utf-8"
)
pq.write_table(pa.Table.from_pandas(snapshot_before_df, preserve_index=False), SOURCE_SNAPSHOT_BEFORE_PARQUET, compression="zstd")

source_sha_map = {row["file"]: row["sha256"] for row in snapshot_before}
current_state_signature = {
    "input_dir": str(INPUT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "survivor_policy": SURVIVOR_POLICY,
    "source_sha256": source_sha_map,
}

existing_output_parts = sorted(OUTPUT_DIR.glob("part_*.parquet"))
if RESUME_EXISTING_OUTPUT and existing_output_parts:
    assert RUN_STATE_PATH.exists(), (
        "Resume mode found existing output parts but no run_state.json. "
        "Refusing to infer whether those outputs belong to this source snapshot."
    )
    saved_state = json.loads(RUN_STATE_PATH.read_text(encoding="utf-8"))
    for key in ("input_dir", "output_dir", "survivor_policy", "source_sha256"):
        assert saved_state.get(key) == current_state_signature[key], (
            f"Resume state mismatch for {key}; refusing to mix runs."
        )
else:
    state_to_save = {
        **current_state_signature,
        "status": "in_progress",
        "run_started_at": RUN_STARTED_AT,
    }
    RUN_STATE_PATH.write_text(json.dumps(state_to_save, indent=2), encoding="utf-8")

print(snapshot_before_df[["part_number", "rows", "size_bytes", "sha256"]].to_string(index=False))
print("Strong pre-run source snapshot: PASS")


SHA-256 source snapshot:   0%|          | 0/57 [00:00<?, ?it/s]

 part_number   rows  size_bytes                                                           sha256
           0 278106  1899045009 65e79cae83931ef3b98a6c43d246ab3a515941bc583e5de852c17368f9ebcaa0
           1 374529  2550107485 ac36239eeb040bab83dadcd29ee956bee496f80331a78f1c6823f82341e9bd62
           2 268518  1728088552 0948d8efa61f2d62ade6a6a0deeab107932256ce7e2ead0fa3783262e5fb1029
           3 342427  2255059773 c964efa3f8cb0265540666b6c9abd3d9ecdc1d19bef59db9657992777ebe040d
           4 259925  2108802133 93d1ba4c29700eafc7a098fb6fb54231ace881cef137b6005e9165bc972c7007
           5 288089  2673303001 14bfd2d0241874f987a92e6319371c1dc93bb10a97bca61b48505767a4f6e883
           6 288631  2657432853 efb006a0c5aa7fde221751c844b30d8e0ed9db911d602ee00a86d2595e98877e
           7 295847  2741096826 239ac885736963f5efefa76ac1f678da41af9d31c705273a162cfae68b2efd6f
           8 267184  2435898446 2e257dcda957a4b53e190461b64ad6b537e3ec5ea379653510ada061ac89776c
           9 277099  262297862

## Cell 6 — Global existing-hash integrity scan

**Purpose**

Treat all 57 Parquet files as one logical dataset and validate the persisted `hash` column before any row is removed.

The server-side diagnostic for this dataset established that:

- the dataset contains 21,087,435 rows;
- there are 6,008,596 globally unique stored hashes;
- there are no NULL hashes;
- every observed stored hash has length 64.

This notebook does **not** infer the hash algorithm from the 64-character representation. Exact deduplication only requires the persisted `hash` values to be stable equality keys. Therefore this cell validates the properties needed by the current client task without imposing the previous notebook's 40-character SHA-1 contract.

The scan establishes the authoritative input row count, global unique-hash count, NULL/empty-key counts, and observed hash-length range.

**Reference**

- Dataset evidence: server-side diagnostic supplied for `/data/joeyllm_data/AUTokens50_with_hash_simhash`.
- Task constraint: exact duplicates are determined by equality of the existing `hash` column.
- DuckDB projection pushdown for Parquet scans: https://duckdb.org/docs/current/data/parquet/overview


In [7]:
# Purpose: Validate the persisted hash column globally without assuming a hash algorithm.
# Reference: Server-side dataset diagnostics; task-defined equality on the existing hash column.

hash_integrity = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN {HASH_COL} IS NULL THEN 1 ELSE 0 END) AS null_hashes,
        SUM(CASE WHEN {HASH_COL} IS NOT NULL AND length({HASH_COL}) = 0 THEN 1 ELSE 0 END) AS empty_hashes,
        MIN(length({HASH_COL})) AS min_hash_length,
        MAX(length({HASH_COL})) AS max_hash_length,
        COUNT(DISTINCT {HASH_COL}) AS unique_hashes
    FROM {INPUT_SCAN_SQL}
""").fetchdf().iloc[0].to_dict()

for key in hash_integrity:
    if pd.notna(hash_integrity[key]):
        hash_integrity[key] = int(hash_integrity[key])

metadata_input_rows = int(schema_df["rows"].sum())

assert hash_integrity["total_rows"] == metadata_input_rows, (
    f"Global scan rows {hash_integrity['total_rows']} != Parquet metadata rows {metadata_input_rows}"
)
assert hash_integrity["null_hashes"] == 0, (
    f"Found {hash_integrity['null_hashes']} NULL hashes. Stop rather than treating NULL as a duplicate key."
)
assert hash_integrity["empty_hashes"] == 0, (
    f"Found {hash_integrity['empty_hashes']} empty-string hashes. Stop rather than treating an empty key as valid."
)

# The following values are observed properties of the current dataset, not an inferred algorithm contract.
print(json.dumps(hash_integrity, indent=2))
print(
    "Observed persisted hash length range:",
    hash_integrity["min_hash_length"],
    "to",
    hash_integrity["max_hash_length"],
)
print("Global existing-hash integrity scan: PASS")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "total_rows": 21087435,
  "null_hashes": 0,
  "empty_hashes": 0,
  "min_hash_length": 64,
  "max_hash_length": 64,
  "unique_hashes": 6008596
}
Observed persisted hash length range: 64 to 64
Global existing-hash integrity scan: PASS


## Cell 7 — Discover all global exact duplicate hashes

**Purpose**

Identify every `hash` that occurs more than once across the complete 57-file dataset. This is a **global** aggregation, so duplicates spanning different `part_*.parquet` files are detected.

The result is persisted as:

```text
~/hash_output/_audit/duplicate_hashes.parquet
```

with one row per duplicate hash and its total occurrence count.

**Reference**

- Lee et al., 2022 demonstrates the importance of deduplication for language-model training data: https://aclanthology.org/2022.acl-long.577/
- Penedo et al., 2024 (FineWeb) documents deduplication as a first-class pretraining-data curation decision and evaluates its effects separately from other filters: https://arxiv.org/abs/2406.17557
- DuckDB multi-file Parquet scan and projection pushdown: https://duckdb.org/docs/current/data/parquet/overview


In [8]:
# Purpose: Find every exact duplicate hash globally across all 57 source Parquet files.
# Reference: Lee et al. 2022; FineWeb 2024; DuckDB multi-file Parquet scanning.

DUPLICATE_HASHES_PATH = AUDIT_DIR / "duplicate_hashes.parquet"
DUPLICATE_HASHES_PATH.unlink(missing_ok=True)

con.execute("DROP TABLE IF EXISTS duplicate_hashes")
con.execute(f"""
    CREATE TABLE duplicate_hashes AS
    SELECT
        {HASH_COL} AS hash,
        COUNT(*)::UBIGINT AS occurrence_count
    FROM {INPUT_SCAN_SQL}
    GROUP BY {HASH_COL}
    HAVING COUNT(*) > 1
    ORDER BY occurrence_count DESC, hash ASC
""")

con.execute(
    f"COPY duplicate_hashes TO {sql_literal(str(DUPLICATE_HASHES_PATH))} "
    "(FORMAT PARQUET, COMPRESSION ZSTD)"
)

duplicate_stats = con.execute("""
    SELECT
        COUNT(*) AS duplicate_hash_groups,
        COALESCE(SUM(occurrence_count), 0) AS rows_in_duplicate_groups,
        COALESCE(SUM(occurrence_count - 1), 0) AS duplicate_rows_to_remove,
        COALESCE(MAX(occurrence_count), 0) AS max_duplicate_group_size
    FROM duplicate_hashes
""").fetchdf().iloc[0].to_dict()

duplicate_stats = {k: int(v) for k, v in duplicate_stats.items()}
print(json.dumps(duplicate_stats, indent=2))
print("Duplicate-hash audit file:", DUPLICATE_HASHES_PATH)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "duplicate_hash_groups": 3147386,
  "rows_in_duplicate_groups": 18226225,
  "duplicate_rows_to_remove": 15078839,
  "max_duplicate_group_size": 103
}
Duplicate-hash audit file: /home/jovyan/hash_output/_audit/duplicate_hashes.parquet


## Cell 8 — Build deterministic survivor and removal indexes

**Purpose**

Assign a deterministic rank to every member of every duplicate-hash group using the accepted survivor policy:

```text
part_number ASC, file_row_number ASC
```

For each duplicate hash:

- rank 1 is the survivor;
- ranks 2..N are removed.

The notebook persists both sides of the decision:

```text
~/hash_output/_audit/duplicate_survivors.parquet
~/hash_output/_audit/removed_rows.parquet
```

`removed_rows.parquet` records the removed row and the exact survivor that caused the removal decision, providing row-level lineage.

**Reference**

- Wang et al., 2026 emphasizes execution provenance and process-level accountability for auditable agent workflows: https://arxiv.org/abs/2606.04990
- The deterministic ordering is an engineering policy for this task; it is not inferred from a paper and does not claim that an earlier row has higher semantic quality.


In [9]:
# Purpose: Create reproducible survivor/removal decisions with row-level provenance.
# Reference: Wang et al. 2026 for provenance/audit principles; deterministic policy is task engineering.

DUPLICATE_SURVIVORS_PATH = AUDIT_DIR / "duplicate_survivors.parquet"
REMOVED_ROWS_PATH = AUDIT_DIR / "removed_rows.parquet"
DUPLICATE_SURVIVORS_PATH.unlink(missing_ok=True)
REMOVED_ROWS_PATH.unlink(missing_ok=True)

con.execute("DROP TABLE IF EXISTS duplicate_members_ranked")
con.execute("DROP TABLE IF EXISTS duplicate_survivors")
con.execute("DROP TABLE IF EXISTS removed_rows")

con.execute(f"""
    CREATE TABLE duplicate_members_ranked AS
    SELECT
        s.{HASH_COL} AS hash,
        s.filename AS source_file,
        m.part_number AS part_number,
        s.file_row_number AS file_row_number,
        ROW_NUMBER() OVER (
            PARTITION BY s.{HASH_COL}
            ORDER BY m.part_number ASC, s.file_row_number ASC
        ) AS occurrence_rank
    FROM {INPUT_SCAN_SQL} AS s
    INNER JOIN source_map AS m
        ON s.filename = m.source_file
    INNER JOIN duplicate_hashes AS d
        ON s.{HASH_COL} = d.hash
""")

con.execute("""
    CREATE TABLE duplicate_survivors AS
    SELECT
        hash,
        source_file AS kept_source_file,
        part_number AS kept_part_number,
        file_row_number AS kept_file_row_number
    FROM duplicate_members_ranked
    WHERE occurrence_rank = 1
    ORDER BY kept_part_number, kept_file_row_number
""")

con.execute("""
    CREATE TABLE removed_rows AS
    SELECT
        r.hash,
        r.source_file AS removed_source_file,
        r.part_number AS removed_part_number,
        r.file_row_number AS removed_file_row_number,
        s.kept_source_file,
        s.kept_part_number,
        s.kept_file_row_number
    FROM duplicate_members_ranked AS r
    INNER JOIN duplicate_survivors AS s
        ON r.hash = s.hash
    WHERE r.occurrence_rank > 1
    ORDER BY r.part_number, r.file_row_number
""")

con.execute(
    f"COPY duplicate_survivors TO {sql_literal(str(DUPLICATE_SURVIVORS_PATH))} "
    "(FORMAT PARQUET, COMPRESSION ZSTD)"
)
con.execute(
    f"COPY removed_rows TO {sql_literal(str(REMOVED_ROWS_PATH))} "
    "(FORMAT PARQUET, COMPRESSION ZSTD)"
)

survivor_count = int(con.execute("SELECT COUNT(*) FROM duplicate_survivors").fetchone()[0])
removed_count = int(con.execute("SELECT COUNT(*) FROM removed_rows").fetchone()[0])
expected_removed_count = int(duplicate_stats["duplicate_rows_to_remove"])
duplicate_group_count = int(duplicate_stats["duplicate_hash_groups"])

assert survivor_count == duplicate_group_count, (
    f"Expected one survivor per duplicate hash group: survivors={survivor_count}, groups={duplicate_group_count}"
)
assert removed_count == expected_removed_count, (
    f"Removal index mismatch: removed_rows={removed_count}, expected={expected_removed_count}"
)

print("Duplicate groups :", duplicate_group_count)
print("Survivors        :", survivor_count)
print("Rows to remove   :", removed_count)
print("Survivor audit   :", DUPLICATE_SURVIVORS_PATH)
print("Removal audit    :", REMOVED_ROWS_PATH)
print("Deterministic survivor/removal indexes: PASS")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate groups : 3147386
Survivors        : 3147386
Rows to remove   : 15078839
Survivor audit   : /home/jovyan/hash_output/_audit/duplicate_survivors.parquet
Removal audit    : /home/jovyan/hash_output/_audit/removed_rows.parquet
Deterministic survivor/removal indexes: PASS


## Cell 9 — Validate the exact-hash deduplication decision contract

**Purpose**

Validate the deduplication plan using only the authoritative persisted `hash` equality relation.

The client-defined scope for this stage is exact-hash deduplication:

> rows with the same existing `hash` are treated as exact duplicates.

A previous notebook version added an extra full-corpus `GROUP BY(hash, text)` guard. That operation required regrouping large text values for more than 18 million duplicate-group rows and is not required by the client task. It is intentionally removed here.

This cell instead proves the decision-plan invariants:

1. every row in `duplicate_hashes` has `occurrence_count >= 2`;
2. `duplicate_members_ranked` contains exactly the rows belonging to duplicate groups;
3. there is exactly one survivor per duplicate hash;
4. `survivors + removals = rows_in_duplicate_groups`;
5. `input_rows - removals = global_unique_hashes`.

No `text`, `hash`, or `simhash` value is modified.

**Reference**

- Task constraint: exact duplicates in this stage are determined by equality of the existing `hash` column.
- Lee et al., 2022, *Deduplicating Training Data Makes Language Models Better*: https://aclanthology.org/2022.acl-long.577/
- FineWeb 2024 treats deduplication as an explicit, separately auditable data-curation stage: https://arxiv.org/abs/2406.17557


In [ ]:
# Purpose: Prove the exact-hash survivor/removal plan is internally complete and consistent.
# Reference: Task-defined equality on the existing hash column; Lee et al. 2022; FineWeb 2024.

DEDUP_CONTRACT_PATH = AUDIT_DIR / "dedup_contract_validation.json"

invalid_duplicate_groups = int(con.execute("""
    SELECT COUNT(*)
    FROM duplicate_hashes
    WHERE occurrence_count < 2
""").fetchone()[0])

ranked_member_count = int(
    con.execute("SELECT COUNT(*) FROM duplicate_members_ranked").fetchone()[0]
)
survivor_count_check = int(
    con.execute("SELECT COUNT(*) FROM duplicate_survivors").fetchone()[0]
)
removed_count_check = int(
    con.execute("SELECT COUNT(*) FROM removed_rows").fetchone()[0]
)

rows_in_duplicate_groups = int(duplicate_stats["rows_in_duplicate_groups"])
duplicate_group_count_check = int(duplicate_stats["duplicate_hash_groups"])
input_rows_check = int(hash_integrity["total_rows"])
unique_hashes_check = int(hash_integrity["unique_hashes"])

assert invalid_duplicate_groups == 0, (
    f"Found {invalid_duplicate_groups} rows in duplicate_hashes with occurrence_count < 2."
)
assert ranked_member_count == rows_in_duplicate_groups, (
    f"Ranked duplicate members mismatch: ranked={ranked_member_count}, "
    f"expected={rows_in_duplicate_groups}"
)
assert survivor_count_check == duplicate_group_count_check, (
    f"Survivor count mismatch: survivors={survivor_count_check}, "
    f"duplicate_groups={duplicate_group_count_check}"
)
assert survivor_count_check + removed_count_check == rows_in_duplicate_groups, (
    f"Duplicate partition mismatch: survivors={survivor_count_check}, "
    f"removed={removed_count_check}, group_rows={rows_in_duplicate_groups}"
)
assert input_rows_check - removed_count_check == unique_hashes_check, (
    f"Global exact-hash conservation failed: input={input_rows_check}, "
    f"removed={removed_count_check}, unique_hashes={unique_hashes_check}"
)

dedup_contract = {
    "status": "pass",
    "decision_key": HASH_COL,
    "decision_rule": "rows with identical persisted hash values are exact duplicates for this stage",
    "invalid_duplicate_groups": invalid_duplicate_groups,
    "rows_in_duplicate_groups": rows_in_duplicate_groups,
    "ranked_duplicate_members": ranked_member_count,
    "duplicate_hash_groups": duplicate_group_count_check,
    "survivors": survivor_count_check,
    "rows_to_remove": removed_count_check,
    "input_rows": input_rows_check,
    "input_unique_hashes": unique_hashes_check,
    "full_text_regrouping_performed": False,
}

DEDUP_CONTRACT_PATH.write_text(
    json.dumps(dedup_contract, indent=2),
    encoding="utf-8",
)

print(json.dumps(dedup_contract, indent=2))
print("Exact-hash deduplication decision contract: PASS")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Cell 10 — Materialize one removal index per source part

**Purpose**

Convert the global row-level removal table into 57 small, exact per-part removal-index Parquet files under:

```text
~/hash_output/_tmp/removal_index
```

Each index contains only the DuckDB `file_row_number` values that must be removed from that exact part. This avoids repeatedly scanning the global removal manifest during the parallel rewrite stage.

**Reference**

- Engineering optimization derived from the global two-phase design: analyze globally once, then rewrite files independently.
- Data-Juicer 2.0 motivates decoupling data-analysis decisions from scalable execution: https://arxiv.org/abs/2501.14755


In [ ]:
# Purpose: Build compact per-part removal indexes for independent parallel rewrites.
# Reference: Two-phase engineering design; Data-Juicer 2.0 scalable execution principles.

# Remove only notebook-owned temporary removal-index files from the current plan.
for old_index in REMOVAL_INDEX_DIR.glob("part_*.parquet"):
    old_index.unlink()

removal_counts_by_part = {}

for part_id in tqdm(EXPECTED_PART_IDS, desc="Materializing removal indexes"):
    table = con.execute(
        """
        SELECT removed_file_row_number AS file_row_number
        FROM removed_rows
        WHERE removed_part_number = ?
        ORDER BY removed_file_row_number
        """,
        [int(part_id)],
    ).fetch_arrow_table()

    index_path = REMOVAL_INDEX_DIR / f"part_{part_id}.parquet"
    pq.write_table(table, index_path, compression="zstd")
    removal_counts_by_part[part_id] = int(table.num_rows)

assert sum(removal_counts_by_part.values()) == removed_count, (
    "Per-part removal indexes do not sum to the global removed-row count."
)

removal_counts_df = pd.DataFrame([
    {"part_number": part_id, "rows_to_remove": removal_counts_by_part[part_id]}
    for part_id in EXPECTED_PART_IDS
])
print(removal_counts_df.to_string(index=False))
print("Per-part removal indexes: PASS")


## Cell 11 — Parallel, Arrow-preserving cleaned-dataset rewrite

**Purpose**

Rewrite the 57 input Parquet files into `~/hash_output` while removing only the row numbers listed in each exact removal index.

Important properties:

- input files are opened for reading only;
- no source file is renamed, unlinked, overwritten, or modified;
- every original column is preserved;
- `text`, `hash`, and `simhash` values are not recalculated or transformed;
- files with zero removals are byte-copied to the output directory;
- files with removals are streamed in Arrow batches and filtered by exact row number;
- up to 4 parts are rewritten concurrently with `ThreadPoolExecutor`;
- Arrow batch decoding inside each rewrite worker is single-threaded to avoid nested CPU oversubscription;
- each generated file is first written to a `.partial` path and atomically promoted with `os.replace()` only after its row count and schema validate.

**Reference**

- Chen et al., 2025, Data-Juicer 2.0: scalable parallel foundation-model data processing. https://arxiv.org/abs/2501.14755
- Apache Arrow / PyArrow Parquet API: https://arrow.apache.org/docs/python/parquet.html
- The exact-removal logic is deterministic engineering; no LLM, SimHash, MinHash, or semantic threshold participates in row deletion.


In [ ]:
# Purpose: Filter exact duplicate rows and write the cleaned dataset in parallel without modifying source files.
# Reference: Data-Juicer 2.0 for scalable processing; PyArrow Parquet for schema-preserving streaming I/O.


def output_is_complete(output_path: Path, expected_rows: int, expected_schema: pa.Schema) -> bool:
    if not output_path.is_file():
        return False
    try:
        pf = pq.ParquetFile(output_path)
        return (
            int(pf.metadata.num_rows) == int(expected_rows)
            and pf.schema_arrow.equals(expected_schema, check_metadata=True)
        )
    except Exception:
        return False


def rewrite_one_part(part_id: int) -> dict:
    input_path = EXPECTED_INPUT_FILES[part_id]
    output_path = OUTPUT_DIR / input_path.name
    partial_path = OUTPUT_DIR / f".{input_path.name}.partial"
    removal_index_path = REMOVAL_INDEX_DIR / f"part_{part_id}.parquet"

    source_pf = pq.ParquetFile(input_path)
    source_schema = source_pf.schema_arrow
    input_rows = int(source_pf.metadata.num_rows)

    removal_table = pq.read_table(removal_index_path, columns=["file_row_number"])
    if removal_table.num_rows:
        removal_ids = np.asarray(removal_table.column("file_row_number").to_numpy(zero_copy_only=False), dtype=np.int64)
        removal_ids = np.unique(removal_ids)
    else:
        removal_ids = np.empty(0, dtype=np.int64)

    assert len(removal_ids) == removal_counts_by_part[part_id], (
        f"Removal-index uniqueness mismatch for part_{part_id}: "
        f"unique={len(removal_ids)}, planned={removal_counts_by_part[part_id]}"
    )

    if len(removal_ids):
        minimum_valid = int(FILE_ROW_NUMBER_BASE)
        maximum_valid = int(FILE_ROW_NUMBER_BASE + input_rows - 1)
        assert int(removal_ids[0]) >= minimum_valid
        assert int(removal_ids[-1]) <= maximum_valid

    expected_rows = input_rows - len(removal_ids)

    if RESUME_EXISTING_OUTPUT and output_is_complete(output_path, expected_rows, source_schema):
        return {
            "part_number": part_id,
            "status": "skipped_complete",
            "input_rows": input_rows,
            "removed_rows": int(len(removal_ids)),
            "output_rows": expected_rows,
            "output_file": str(output_path),
        }

    partial_path.unlink(missing_ok=True)

    if len(removal_ids) == 0:
        shutil.copy2(input_path, partial_path)
    else:
        writer = None
        rows_seen = 0
        try:
            writer = pq.ParquetWriter(
                partial_path,
                source_schema,
                compression=PARQUET_COMPRESSION,
                use_dictionary=True,
                write_statistics=True,
            )

            for batch in source_pf.iter_batches(batch_size=REWRITE_BATCH_SIZE, use_threads=False):
                batch_rows = batch.num_rows
                row_numbers = np.arange(
                    rows_seen + FILE_ROW_NUMBER_BASE,
                    rows_seen + FILE_ROW_NUMBER_BASE + batch_rows,
                    dtype=np.int64,
                )
                keep_mask = ~np.isin(row_numbers, removal_ids, assume_unique=True)

                if keep_mask.any():
                    batch_table = pa.Table.from_batches([batch], schema=source_schema)
                    filtered_table = batch_table.filter(pa.array(keep_mask))
                    if filtered_table.num_rows:
                        writer.write_table(filtered_table)

                rows_seen += batch_rows

            assert rows_seen == input_rows, (
                f"Read-row mismatch for {input_path}: rows_seen={rows_seen}, expected={input_rows}"
            )
        finally:
            if writer is not None:
                writer.close()

    produced_pf = pq.ParquetFile(partial_path)
    assert int(produced_pf.metadata.num_rows) == expected_rows, (
        f"Output-row mismatch for {input_path.name}: "
        f"produced={produced_pf.metadata.num_rows}, expected={expected_rows}"
    )
    assert produced_pf.schema_arrow.equals(source_schema, check_metadata=True), (
        f"Output schema mismatch for {input_path.name}"
    )

    os.replace(partial_path, output_path)

    return {
        "part_number": part_id,
        "status": "written",
        "input_rows": input_rows,
        "removed_rows": int(len(removal_ids)),
        "output_rows": expected_rows,
        "output_file": str(output_path),
    }


rewrite_results = []
with ThreadPoolExecutor(max_workers=REWRITE_WORKERS) as executor:
    futures = {executor.submit(rewrite_one_part, part_id): part_id for part_id in EXPECTED_PART_IDS}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Parallel cleaned rewrite"):
        result = future.result()
        rewrite_results.append(result)
        print(
            f"part_{result['part_number']} | {result['status']} | "
            f"input={result['input_rows']:,} | removed={result['removed_rows']:,} | "
            f"output={result['output_rows']:,}",
            flush=True,
        )

rewrite_results = sorted(rewrite_results, key=lambda x: x["part_number"])
rewrite_df = pd.DataFrame(rewrite_results)

assert int(rewrite_df["removed_rows"].sum()) == removed_count
print("\nParallel rewrite completed.")
print(rewrite_df.to_string(index=False))


## Cell 12 — Per-file structural validation

**Purpose**

Validate all 57 cleaned Parquet files independently after the rewrite.

For every part, the cell proves:

- the output file exists;
- output schema is exactly equal to the source Arrow schema, including metadata;
- `output_rows = input_rows - removed_rows_for_this_part`;
- all 57 expected output files exist and no extra `part_*.parquet` files were produced.

**Reference**

- Engineering validation derived directly from the task constraints: this is a row-filtering transformation, not a schema transformation.
- Apache Arrow schema/Parquet metadata APIs: https://arrow.apache.org/docs/python/parquet.html


In [ ]:
# Purpose: Prove all 57 output files are structurally complete and preserve the original schema.
# Reference: Task invariants and Apache Arrow/PyArrow Parquet metadata APIs.

output_part_files = [OUTPUT_DIR / f"part_{i}.parquet" for i in EXPECTED_PART_IDS]
actual_output_part_files = sorted(OUTPUT_DIR.glob("part_*.parquet"), key=lambda p: p.name)

assert all(p.is_file() for p in output_part_files), "One or more expected cleaned part files are missing."
assert {p.resolve() for p in actual_output_part_files} == {p.resolve() for p in output_part_files}, (
    "Output part-file set is not exactly part_0.parquet through part_56.parquet."
)

output_validation_rows = []
for part_id, input_path, output_path in tqdm(
    list(zip(EXPECTED_PART_IDS, EXPECTED_INPUT_FILES, output_part_files)),
    desc="Validating output structure",
):
    src_pf = pq.ParquetFile(input_path)
    out_pf = pq.ParquetFile(output_path)

    expected_rows = int(src_pf.metadata.num_rows) - int(removal_counts_by_part[part_id])
    actual_rows = int(out_pf.metadata.num_rows)

    assert out_pf.schema_arrow.equals(src_pf.schema_arrow, check_metadata=True), (
        f"Schema changed for part_{part_id}.parquet"
    )
    assert actual_rows == expected_rows, (
        f"Row-count mismatch for part_{part_id}.parquet: actual={actual_rows}, expected={expected_rows}"
    )

    output_validation_rows.append({
        "part_number": part_id,
        "input_rows": int(src_pf.metadata.num_rows),
        "removed_rows": int(removal_counts_by_part[part_id]),
        "output_rows": actual_rows,
        "schema_equal": True,
    })

output_validation_df = pd.DataFrame(output_validation_rows)
print(output_validation_df.to_string(index=False))
print("Per-file structural validation: PASS")


## Cell 13 — Global output deduplication invariants

**Purpose**

Treat the cleaned 57-file output as one logical dataset and prove the global exact-deduplication result.

Required invariants:

1. `input_rows = output_rows + removed_rows`;
2. `input_unique_hashes = output_unique_hashes`;
3. `output_rows = output_unique_hashes`, so every output hash occurs exactly once;
4. no output hash has `COUNT(*) > 1`;
5. the output `hash` column remains non-NULL and non-empty;
6. the observed output hash-length range is recorded and compared with the input range.

No assumption is made here about whether the 64-character persisted hash is SHA-256 or another representation.

**Reference**

- Lee et al., 2022: deduplication should explicitly remove repeated training examples rather than merely annotate them. https://aclanthology.org/2022.acl-long.577/
- FineWeb 2024: data-curation transformations are evaluated as distinct, auditable stages. https://arxiv.org/abs/2406.17557
- DuckDB multi-file Parquet scan: https://duckdb.org/docs/current/data/parquet/overview


In [ ]:
# Purpose: Prove that the output has exactly one row per existing hash and satisfies row-count invariants.
# Reference: Lee et al. 2022; FineWeb 2024; DuckDB multi-file Parquet scanning.

OUTPUT_GLOB = str(OUTPUT_DIR / "part_*.parquet")
OUTPUT_SCAN_SQL = (
    f"read_parquet({sql_literal(OUTPUT_GLOB)}, "
    "filename=true, file_row_number=true, hive_partitioning=false, union_by_name=false)"
)

output_global = con.execute(f"""
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT {HASH_COL}) AS output_unique_hashes,
        SUM(CASE WHEN {HASH_COL} IS NULL THEN 1 ELSE 0 END) AS null_hashes,
        SUM(CASE WHEN {HASH_COL} IS NOT NULL AND length({HASH_COL}) = 0 THEN 1 ELSE 0 END) AS empty_hashes,
        MIN(length({HASH_COL})) AS min_hash_length,
        MAX(length({HASH_COL})) AS max_hash_length
    FROM {OUTPUT_SCAN_SQL}
""").fetchdf().iloc[0].to_dict()

output_global = {
    k: (None if pd.isna(v) else int(v))
    for k, v in output_global.items()
}

remaining_duplicate_groups = int(con.execute(f"""
    SELECT COUNT(*)
    FROM (
        SELECT {HASH_COL}
        FROM {OUTPUT_SCAN_SQL}
        GROUP BY {HASH_COL}
        HAVING COUNT(*) > 1
    )
""").fetchone()[0])

input_rows = int(hash_integrity["total_rows"])
input_unique_hashes = int(hash_integrity["unique_hashes"])
output_rows = int(output_global["output_rows"])
output_unique_hashes = int(output_global["output_unique_hashes"])

assert input_rows == output_rows + removed_count, (
    f"Row conservation failed: input={input_rows}, output={output_rows}, removed={removed_count}"
)
assert input_unique_hashes == output_unique_hashes, (
    f"Unique-hash conservation failed: input={input_unique_hashes}, output={output_unique_hashes}"
)
assert output_rows == output_unique_hashes, (
    f"Output is not fully exact-deduplicated: rows={output_rows}, unique_hashes={output_unique_hashes}"
)
assert remaining_duplicate_groups == 0, (
    f"Found {remaining_duplicate_groups} duplicate hash groups in the cleaned output."
)
assert output_global["null_hashes"] == 0, "NULL hash appeared in cleaned output."
assert output_global["empty_hashes"] == 0, "Empty-string hash appeared in cleaned output."
assert output_global["min_hash_length"] == hash_integrity["min_hash_length"], (
    "Minimum persisted hash length changed between input and output."
)
assert output_global["max_hash_length"] == hash_integrity["max_hash_length"], (
    "Maximum persisted hash length changed between input and output."
)

print(json.dumps(output_global, indent=2))
print("Remaining duplicate hash groups:", remaining_duplicate_groups)
print("Global output exact-deduplication invariants: PASS")


## Cell 14 — Post-run source immutability proof

**Purpose**

Recompute the same metadata and SHA-256 checksums for all 57 source files after the cleaned dataset has been created, then compare them with the pre-run snapshot.

The notebook requires exact equality for:

- path;
- row count;
- byte size;
- modification time;
- SHA-256 checksum.

This provides byte-level evidence that the source dataset remained unchanged during the run.

**Reference**

- Wang et al., 2026: provenance and process-level accountability for auditable workflows. https://arxiv.org/abs/2606.04990
- Cryptographic integrity comparison is an engineering control applied to the immutable-source requirement.


In [ ]:
# Purpose: Prove byte-for-byte that no source Parquet file changed during deduplication.
# Reference: 2026 provenance/audit principles plus cryptographic file-integrity checking.

SOURCE_SNAPSHOT_AFTER_JSON = AUDIT_DIR / "source_snapshot_after.json"
SOURCE_SNAPSHOT_AFTER_PARQUET = AUDIT_DIR / "source_snapshot_after.parquet"

snapshot_after = []
with ThreadPoolExecutor(max_workers=CHECKSUM_WORKERS) as executor:
    futures = {
        executor.submit(snapshot_one, part_id, path): part_id
        for part_id, path in zip(EXPECTED_PART_IDS, EXPECTED_INPUT_FILES)
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Post-run SHA-256 source snapshot"):
        snapshot_after.append(future.result())

snapshot_after = sorted(snapshot_after, key=lambda x: x["part_number"])
snapshot_after_df = pd.DataFrame(snapshot_after)

SOURCE_SNAPSHOT_AFTER_JSON.write_text(
    json.dumps(snapshot_after, indent=2), encoding="utf-8"
)
pq.write_table(pa.Table.from_pandas(snapshot_after_df, preserve_index=False), SOURCE_SNAPSHOT_AFTER_PARQUET, compression="zstd")

before_by_part = {row["part_number"]: row for row in snapshot_before}
after_by_part = {row["part_number"]: row for row in snapshot_after}
assert set(before_by_part) == set(after_by_part) == set(EXPECTED_PART_IDS)

comparison_fields = ["file", "rows", "size_bytes", "mtime_ns"]
if VERIFY_SOURCE_FILE_SHA256:
    comparison_fields.append("sha256")

source_changes = []
for part_id in EXPECTED_PART_IDS:
    before = before_by_part[part_id]
    after = after_by_part[part_id]
    changed_fields = [field for field in comparison_fields if before[field] != after[field]]
    if changed_fields:
        source_changes.append({
            "part_number": part_id,
            "changed_fields": changed_fields,
            "before": before,
            "after": after,
        })

assert not source_changes, (
    "Source immutability verification failed. One or more input files changed: "
    + json.dumps(source_changes, indent=2)
)

print("Compared fields:", comparison_fields)
print("Changed source files: 0")
print("Source immutability proof: PASS")


## Cell 15 — Final audit summary and run completion state

**Purpose**

Write a compact machine-readable summary of the completed exact-deduplication run and mark the run state as `completed` only after every validation has passed.

The summary records the exact paths, versions, row counts, duplicate statistics, survivor policy, validation results, audit artifacts, and source-immutability result.

**Reference**

- Wang et al., 2026 motivates preserving execution provenance rather than relying only on the final artifact: https://arxiv.org/abs/2606.04990
- FineWeb 2024 and Data-Juicer 2.0 both emphasize explicit, reproducible data-processing stages for foundation-model datasets: https://arxiv.org/abs/2406.17557 and https://arxiv.org/abs/2501.14755


In [ ]:
# Purpose: Persist the final validated run summary and mark the provenance state as completed.
# Reference: Provenance/audit literature; FineWeb and Data-Juicer reproducible data-processing practice.

SUMMARY_PATH = AUDIT_DIR / "summary.json"
PER_FILE_STATS_PATH = AUDIT_DIR / "per_file_stats.parquet"

pq.write_table(
    pa.Table.from_pandas(output_validation_df, preserve_index=False),
    PER_FILE_STATS_PATH,
    compression="zstd",
)

run_finished_at = datetime.now(timezone.utc).isoformat()
elapsed_seconds = time.time() - RUN_START_WALL

summary = {
    "status": "completed",
    "run_started_at": RUN_STARTED_AT,
    "run_finished_at": run_finished_at,
    "elapsed_seconds": elapsed_seconds,
    "input_dir": str(INPUT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "input_file_count": 57,
    "output_file_count": 57,
    "input_rows": input_rows,
    "output_rows": output_rows,
    "input_unique_hashes": input_unique_hashes,
    "output_unique_hashes": output_unique_hashes,
    "duplicate_hash_groups": duplicate_stats["duplicate_hash_groups"],
    "rows_in_duplicate_groups": duplicate_stats["rows_in_duplicate_groups"],
    "duplicate_rows_removed": removed_count,
    "max_duplicate_group_size": duplicate_stats["max_duplicate_group_size"],
    "remaining_duplicate_hash_groups": remaining_duplicate_groups,
    "survivor_policy": SURVIVOR_POLICY,
    "duckdb_file_row_number_base": int(FILE_ROW_NUMBER_BASE),
    "source_immutable": True,
    "source_sha256_verified": bool(VERIFY_SOURCE_FILE_SHA256),
    "schema_preserved": True,
    "hash_column_recomputed_or_modified": False,
    "observed_input_hash_length_min": hash_integrity["min_hash_length"],
    "observed_input_hash_length_max": hash_integrity["max_hash_length"],
    "simhash_column_recomputed_or_modified": False,
    "duckdb_threads": int(DUCKDB_THREADS),
    "duckdb_memory_limit": DUCKDB_MEMORY_LIMIT,
    "rewrite_workers": int(REWRITE_WORKERS),
    "duckdb_version": duckdb.__version__,
    "pyarrow_version": pa.__version__,
    "audit_artifacts": {
        "duplicate_hashes": str(DUPLICATE_HASHES_PATH),
        "duplicate_survivors": str(DUPLICATE_SURVIVORS_PATH),
        "removed_rows": str(REMOVED_ROWS_PATH),
        "dedup_contract_validation": str(DEDUP_CONTRACT_PATH),
        "per_file_stats": str(PER_FILE_STATS_PATH),
        "source_snapshot_before": str(SOURCE_SNAPSHOT_BEFORE_PARQUET),
        "source_snapshot_after": str(SOURCE_SNAPSHOT_AFTER_PARQUET),
    },
    "references": [
        "Lee et al. 2022, https://aclanthology.org/2022.acl-long.577/",
        "Penedo et al. 2024 (FineWeb), https://arxiv.org/abs/2406.17557",
        "Chen et al. 2025 (Data-Juicer 2.0), https://arxiv.org/abs/2501.14755",
        "Wang et al. 2026 (Execution Provenance), https://arxiv.org/abs/2606.04990",
        "DuckDB Parquet documentation, https://duckdb.org/docs/current/data/parquet/overview",
    ],
}

SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

completed_state = {
    "input_dir": str(INPUT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "survivor_policy": SURVIVOR_POLICY,
    "source_sha256": source_sha_map,
    "status": "completed",
    "run_started_at": RUN_STARTED_AT,
    "run_finished_at": run_finished_at,
    "summary": str(SUMMARY_PATH),
}
RUN_STATE_PATH.write_text(json.dumps(completed_state, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
print("\nFINAL RESULT: exact-hash deduplication completed and fully validated.")
print("Cleaned dataset:", OUTPUT_DIR)
print("Audit summary  :", SUMMARY_PATH)


## Completion checklist

**Purpose**

State the exact completion conditions that the preceding code cells enforce with assertions.

- [x] All `part_0.parquet` through `part_56.parquet` are treated as one dataset for duplicate discovery.
- [x] All exact duplicate `hash` values are discovered globally.
- [x] The exact-hash survivor/removal decision contract is validated without full-text regrouping.
- [x] One deterministic survivor is retained for each duplicate hash.
- [x] Every other member of that duplicate-hash group is removed.
- [x] No SimHash, MinHash, embedding similarity, or LLM judgment is used in this exact-dedup stage.
- [x] All original columns and surviving row values are preserved.
- [x] The immutable source directory is never used as an output target.
- [x] Cleaned data is written only to `~/hash_output/part_*.parquet`.
- [x] Row conservation is proven: `input_rows = output_rows + removed_rows`.
- [x] Unique-hash conservation is proven.
- [x] The output contains exactly one row per hash.
- [x] Row-level survivor/removal provenance is persisted.
- [x] SHA-256 snapshots prove the input files are byte-identical before and after the run.

**Reference**

- Task requirements supplied for this notebook.
- Lee et al., 2022: https://aclanthology.org/2022.acl-long.577/
- FineWeb 2024: https://arxiv.org/abs/2406.17557
- Data-Juicer 2.0: https://arxiv.org/abs/2501.14755
- Wang et al., 2026: https://arxiv.org/abs/2606.04990
